# LeetCode #1145: Binary Tree Coloring Game

https://leetcode.com/problems/binary-tree-coloring-game/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: DFS Subtree Count ★** | $O(n)$ | $O(h)$ |

---

## Understanding the Methods

### Brute Force
Simulate all possible coloring strategies for both players exhaustively, recomputing reachable node counts from scratch for each candidate. Redundant tree traversals make this quadratic.

### Optimal: DFS Subtree Count ★
Player 2 can only block Player 1 by choosing one of three nodes: the left child, the right child, or the parent of `x`. A single DFS computes the size of each subtree rooted at `x`'s children. The parent component size is `n - leftCount - rightCount - 1`. Player 2 wins if any of these three components is strictly larger than `n / 2`.

**Constraints:**
* The number of nodes is $n$
* $1 \leq x \leq n \leq 100$
* $n$ is odd (guarantees no tie)

## Solutions

### C#

In [ ]:
public class Solution {
    private int _leftCount, _rightCount, _target;

    public bool BtreeGameWinningMove(TreeNode root, int n, int x) {
        _target = x;
        // Count nodes in both subtrees of x with one DFS
        Count(root);
        int parent = n - _leftCount - _rightCount - 1;
        // Player 2 wins if any reachable component from x is bigger than half
        return Math.Max(Math.Max(_leftCount, _rightCount), parent) > n / 2;
    }

    private int Count(TreeNode node) {
        if (node == null) return 0;
        int left = Count(node.left), right = Count(node.right);
        // Capture subtree sizes the moment we process x
        if (node.val == _target) { _leftCount = left; _rightCount = right; }
        return left + right + 1;
    }
}

### Python

In [ ]:
class Solution:
    def btree_game_winning_move(self, root, n: int, x: int) -> bool:
        left_count = right_count = 0

        def count(node) -> int:
            nonlocal left_count, right_count
            if not node: return 0
            left = count(node.left)
            right = count(node.right)
            # Capture subtree sizes the moment we process x
            if node.val == x:
                left_count = left
                right_count = right
            return left + right + 1

        count(root)
        parent = n - left_count - right_count - 1
        # Player 2 wins if any reachable component from x is bigger than half
        return max(left_count, right_count, parent) > n // 2

### Go

In [ ]:
func btreeGameWinningMove(root *TreeNode, n int, x int) bool {
    leftCount, rightCount := 0, 0
    var count func(node *TreeNode) int
    count = func(node *TreeNode) int {
        if node == nil { return 0 }
        left := count(node.Left)
        right := count(node.Right)
        // Capture subtree sizes the moment we process x
        if node.Val == x { leftCount = left; rightCount = right }
        return left + right + 1
    }
    count(root)
    parent := n - leftCount - rightCount - 1
    best := leftCount; if rightCount > best { best = rightCount }; if parent > best { best = parent }
    // Player 2 wins if any reachable component from x is bigger than half
    return best > n/2
}

### Rust

In [ ]:
use std::rc::Rc;
use std::cell::RefCell;
impl Solution {
    pub fn btree_game_winning_move(root: Option<Rc<RefCell<TreeNode>>>, n: i32, x: i32) -> bool {
        let mut left_count = 0i32;
        let mut right_count = 0i32;
        Self::count(&root, x, &mut left_count, &mut right_count);
        let parent = n - left_count - right_count - 1;
        // Player 2 wins if any reachable component from x is bigger than half
        left_count.max(right_count).max(parent) > n / 2
    }

    fn count(node: &Option<Rc<RefCell<TreeNode>>>, x: i32, lc: &mut i32, rc: &mut i32) -> i32 {
        if let Some(n) = node {
            let b = n.borrow();
            let left = Self::count(&b.left, x, lc, rc);
            let right = Self::count(&b.right, x, lc, rc);
            // Capture subtree sizes the moment we process x
            if b.val == x { *lc = left; *rc = right; }
            left + right + 1
        } else { 0 }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n = 11`, `x = 3` (balanced binary tree, x is the root's left child)
Left subtree of $x$ has 3 nodes, right subtree has 3 nodes, parent component has $11-3-3-1=4$ nodes. Largest component is $4 > 5$ is false; Player 1 wins.

### 2. Slightly Complex
**Input:** `n = 11`, `x = 3`, but `x` is near a leaf
If the left subtree of `x` is $0$, right is $1$, parent is $9$, then $9 > 5$ — Player 2 picks the parent node and wins.

### 3. Edge Case: Time Factor
**Input:** Linear chain of $n = 99$ nodes, `x = 50` (middle)
The DFS visits all 99 nodes before returning. Left subtree = 49, right subtree = 49, parent = 0. Neither exceeds $49 = n/2$, so Player 1 wins — the balanced chain is unbeatable.

### 4. Edge Case: Space Factor
**Input:** `n = 3`, `x = 1` (root of a 3-node tree)
Minimal tree. DFS call stack depth is 2. Left = right = 1, parent = 0. Largest component is $1 \not > 1 = n/2$; Player 1 wins.

### 5. Almost-Impossible but Plausible
**Input:** `n = 99`, `x = 2` (root's left child; right subtree of root has 97 nodes)
Parent component = $99 - 0 - 0 - 1 = 98$ (if `x` is a leaf). Player 2 picks the root and controls all 98 remaining nodes. $98 > 49$ — Player 2 wins, illustrating that a poorly placed `x` near a leaf surrenders the entire opposite subtree.